<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-LLM-Course/blob/main/05_The_Datasets_Library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

In chapter 3, you got your first taste of the Datasets library and saw that there were three main steps when it came to fine-tuning a model:

  1. Load a dataset from the Hugging Face Hub
  2. Preprocess the data with `Dataset.map()`
  3. Load and compute metrics.

But this is just scratching the surface of what Datasets can do! In this chapter, we will take a deep dive into the library. Along the way, we'll find answers to the following questions:

  - What do you do when you dataset is not on the Hub?
  - How can you slice and dice a dataset? (And what if you *really* need to use Pandas?)
  - What do you do when your dataset is huge and will melt your laptor's RAM?
  - What the heck are "memory mapping" and Apache Arrow?
  - How can you create your own dataset and push it to the Hub?

The techniques you learned here will prepare you for the advanced tokenization and fine-tuning tasks in Chapter 6 and Chapter 7 -so grab a coffee and let's get started!

## What if my dataset isn't on the Hub?

You know how to use the Hugging Face Hub: https://huggingface.co/datasets to download datasets, but you'll often find yourself working with data that is stored either on your laptor or on a remote server. In this section we'll show you how Datasets can be used to load datasets that aren't available on the Hugging Face Hub.

### Working with local and remote datasets

Datasets provides loading scripts to handle the loading of local and remote datasets. It supports several common data formats, such as:  

| Data format | Loading script | Example |
|---|---|---|
| CSV & TSV | `csv` | `load_dataset("csv", data_files="my_file.csv")` |
| Text files | `text` | `load_dataset("text", data_files="my_file.txt")` |
| JSON & JSON Lines | `json` | `load_dataset("json", data_files="my_file.jsonl")` |
| Pickled DataFrames | `pandas` | `load_dataset("pandas", data_files="my_dataframe.pkl")` |

As shown in the table, for each data format we just need to specify the type of loading script in the `load_dataset()` function, along with the `data_files` argument that specifies the path to one or more files. Let's start by loading a dataset from local files; later we'll see how to do the same with remote files.

### Loading a local dataset

For this example we'll use the SQuAd-it daraset: https://github.com/crux82/squad-it/, which is a large-scale dataset for question answering in Italian.

The training and test split are hosted on Github, so we can download them with a simple `wget` command:

In [1]:
!wget https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz
!wget https://github.com/crux82/squad-it/raw/master/SQuAD_it-test.json.gz

--2026-10-18 14:56:15--  https://github.com/crux82/squad-it/raw/master/SQuAD_it-train.json.gz
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/crux82/squad-it/master/SQuAD_it-train.json.gz [following]
--2026-10-18 14:56:17--  https://raw.githubusercontent.com/crux82/squad-it/master/SQuAD_it-train.json.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7725286 (7.4M) [application/octet-stream]
Saving to: ‘SQuAD_it-train.json.gz’

SQuAD_it-train.json 100%[===================>]   7.37M  --.-KB/s    in 0.07s   

2026-10-18 14:56:17 (111 MB/s) - ‘SQuAD_it-train.json.gz’ saved [7725286/7725286]

--2026

This will download two compressed files called `SQuAD_it-train.json.gz` and `AQuAD_it-test.json.gz`, which we can decompress with the Linux `gzip` command:

In [2]:
!gzip -dkv SQuAD_it-*.json.gz

SQuAD_it-test.json.gz:	 87.5% -- created SQuAD_it-test.json
SQuAD_it-train.json.gz:	 82.3% -- created SQuAD_it-train.json


We acn see that the compressed files have been replaced with SQuAD_it-train.json` and `SQuAD_it-test.json`, and that the data is stored in the JSON format.

##